# 💬 03 · RFP 챗봇 — 대화 + 실시간 RAGAS
**`01_rag_build.ipynb` 만 끝나 있으면 바로 실행됩니다.** `02_generate_eval.ipynb` 는 돌리지 않아도 됩니다. 01 이 저장한 인덱스(FAISS · BM25 · 임베딩 모델)를 불러와 **답변 생성만** 합니다. 학습·임베딩은 하지 않습니다.

```
질문(입력 그대로) ─▶ (선택) 기관 라우팅 ─▶ BM25 × Dense(FAISS) 하이브리드 ─▶ gpt-5-mini 답변 ─▶ 실시간 RAGAS (턴마다)
```
- 같은 대화창 안에서는 앞 질문을 기억합니다(멀티턴). **[새 대화]** 를 누르면 처음부터 다시 시작합니다.
- 답변이 나오면 **그 답변의 RAGAS 를 바로 계산**해서 아래 표에 실시간으로 채웁니다. Faithfulness · Answer Relevancy · Context Precision 은 항상, Answer Correctness 는 정답 칸을 채운 질문만 계산합니다.
- 대화와 RAGAS 는 **노트북 화면(메모리)에만** 표시합니다. CSV 등 파일로 저장하지 않습니다.
- **토큰 절약:** `0-2`에서 답변·심판 모델과 **추론 강도(none~max)**를 고르세요. 답변마다 사용 토큰·예상 비용이 표시됩니다. 02 의 골든 데이터셋에 있는 질문은 정답이 자동으로 붙고 **토큰 0 지표**(Key Fact · Semantic Sim)도 계산됩니다.
- 사용 순서: `0-1 → 0-2 → 1-1 → 1-1b → 2-1` 을 실행한 다음, `2-2` 의 `QUESTION` 에 질문을 적고 실행하세요 (다시 실행하면 대화가 이어짐).
- ⚠️ **노트북은 한 번에 하나만 실행하세요** (VM RAM 16GB). 다른 노트북 커널이 남아 있으면 메모리 부족으로 커널이 강제 종료됩니다. `0-1` 셀이 다른 커널과 RAM·GPU 사용량을 보여주고, `CLOSE_OTHER_KERNELS = True` 로 정리할 수 있습니다.
- **🔎 검색 파이프라인(1-1b):** 질문을 받을 때마다 ① GPT 가 검색용 질문으로 해석(추론 강도 low~high · 후속 질문은 직전 대화를 참고) → ② 원래 질문 + 다시 쓴 질문으로 BM25+Dense → RRF → ③ 리랭커 → ④ 답변 — 원래 질문 + 검색용 질문(GPT 해석)을 함께 받아 답함(`ANSWER_WITH_REWRITE`).
- **💵 비용:** 답변마다 질문 해석 · 답변 · RAGAS 심판의 달러 비용을 표시하고, `2-3` 에서 세션 합계와 대화 기록을 보여줍니다.

## 0. 환경 · CPU/GPU · 설정
> 👥 **담당:** 0-1 PM · 0-2 검색 설정 = 검색 · 리랭킹, LLM·질문 해석 설정 = 질문 해석 · 답변

In [1]:
# 0-1) 경로 · 토큰(OpenAI .env / HuggingFace 로그인) · CPU/GPU 버전 확인   — 보통 10~30초
#  ▸ 원격 GCP: g2-standard-4 (vCPU 4 · RAM 16GB) + NVIDIA L4 24GB — VS Code Remote-SSH / JupyterHub 커널에서 실행
#  ▸ HuggingFace 로그인(선택, VM 터미널에서 한 번): hf auth login   ← 이후 HF_TOKEN 입력창 없음
#  ▸ ⑤ 메모리 점검: 다른 노트북 커널이 RAM·GPU 를 잡고 있으면 경고 (CLOSE_OTHER_KERNELS=True 로 종료)
import os, sys, time
from pathlib import Path

T0 = time.time()
PROJECT_DIR = Path.cwd() if (Path.cwd() / "rag").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
if sys.version_info < (3, 10):
    raise RuntimeError(f"Python 3.10 이상이 필요합니다 (현재 {sys.version.split()[0]}) → 커널을 Python 3.10 으로 바꾸세요")

HF_CACHE_DIR = None        # 모델 저장 위치(기본 ~/.cache/huggingface, 약 10GB). 디스크가 부족하면 예) "/mnt/disk/hf_cache"
NEED_OPENAI  = True       # 02: 답변 생성·RAGAS 에 필수

print("① torch 로딩…", flush=True)
from rag.hw import apply_runtime_env, detect, gpu_mem, summary, require_gpu
apply_runtime_env(cache_dir=HF_CACHE_DIR)
print(f"   완료 {time.time() - T0:.1f}s", flush=True)

print("② 토큰 — OpenAI: .env / HuggingFace: 로그인 정보(hf auth login) 사용, HF_TOKEN 입력창 없음", flush=True)
from rag.credentials import load_tokens, verify_tokens, mask
import pandas as pd
tok_status = load_tokens(PROJECT_DIR / ".env", ask=False, save_to_env=False, required=())   # 입력창을 띄우지 않음
_k = os.environ.get("OPENAI_API_KEY", "").strip()
while _k.startswith("sk-sk-"):                    # 'sk-' 가 두 번 붙은 키 바로잡기 (메모리에서만)
    _k = _k[3:]
if _k:
    os.environ["OPENAI_API_KEY"] = _k
elif NEED_OPENAI:
    import getpass
    print("🔑 OPENAI_API_KEY 가 .env 에 없습니다 → 입력창(VS Code: 창 맨 위)에 붙여넣고 Enter", flush=True)
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ").strip()
    tok_status["OPENAI_API_KEY"]["출처"] = "입력(메모리에만)"

# HuggingFace: 터미널에서 한 번 'hf auth login'(구버전: huggingface-cli login) 하면 저장된 로그인 정보를 자동 사용
from huggingface_hub import get_token
_hf = get_token() or next((p.read_text().strip() for p in [Path.home() / ".cache" / "huggingface" / "token"] if p.exists()), None)
if _hf:
    os.environ["HF_TOKEN"] = _hf
oa = os.environ.get("OPENAI_API_KEY", "")
display(pd.DataFrame({
    "OPENAI_API_KEY": {"값": mask(oa), "출처": tok_status["OPENAI_API_KEY"]["출처"] if oa else "-",
                       "용도": "gpt-5-mini 생성·RAGAS 심판" + ("" if NEED_OPENAI else " (01 에서는 파인튜닝 QA 생성 때만)")},
    "HuggingFace": {"값": "로그인됨" if _hf else "로그인 안 됨", "출처": "hf auth login" if _hf else "-",
                    "용도": "모델 다운로드 (KURE-v1·Qwen2.5-VL 은 공개 모델이라 로그인 없이도 받아짐)"},
}).T)
print("③ 토큰 검증 (각 최대 15초)", flush=True)
tok_check = verify_tokens("gpt-5-mini")
print("검증:", tok_check)
if NEED_OPENAI and any(e in tok_check.get("OPENAI_API_KEY", "") for e in ("AuthenticationError", "PermissionDeniedError", "NotFoundError")):
    raise RuntimeError(f"OpenAI 키를 쓸 수 없습니다: {tok_check['OPENAI_API_KEY']} · 현재 키 {mask(os.environ.get('OPENAI_API_KEY'))}"
                       " → .env 의 OPENAI_API_KEY 확인 후 이 셀을 다시 실행하세요")

print("④ GPU 점검", flush=True)
REQUIRE_GPU = False         # 생성은 API, GPU(L4)는 질문 임베딩에만 사용 — 없으면 CPU
HW = detect()
from rag.hw import hw_table
display(hw_table(HW).rename(columns={"값": "지금 사용 중인 CPU · GPU · 버전"}))
if HW.get("gpu"):
    print(("✅ " if HW.get("kernel_ok") else "❌ ") + str(HW.get("kernel_msg")))
print(gpu_mem())
if REQUIRE_GPU:
    require_gpu(HW)
print("⑤ 메모리 점검", flush=True)
# ⑤ 메모리 점검 — VM RAM 16GB: 다른 노트북 커널이 남아 있으면 메모리가 모자라 리눅스가 커널을 강제 종료(OOM)합니다
CLOSE_OTHER_KERNELS = False   # True: 이 노트북 말고 떠 있는 다른 Jupyter 커널을 종료해 RAM·GPU 확보 (그 노트북들의 변수는 사라짐)
import psutil, subprocess
def _gb(b): return round(b / 1024**3, 1)
_me = psutil.Process()
_gpu = {}
try:
    _o = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,used_memory", "--format=csv,noheader,nounits"],
                        capture_output=True, text=True, timeout=10).stdout
    for _ln in _o.strip().splitlines():
        _p, _m = [x.strip() for x in _ln.split(",")]
        _gpu[int(_p)] = round(int(_m) / 1024, 1)
except Exception:
    pass
_others = []
for _p in psutil.process_iter(["pid", "cmdline", "memory_info", "create_time"]):
    try:
        _cl = _p.info["cmdline"] or []
        if (_p.info["pid"] != _me.pid and _cl and "python" in os.path.basename(_cl[0]).lower()
                and any("ipykernel" in a for a in _cl[1:])):          # 다른 노트북 커널 (python -m ipykernel_launcher …)
            _others.append(_p)
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass
if _others:
    display(pd.DataFrame([{"PID": p.pid, "RAM(GB)": _gb(p.info["memory_info"].rss), "GPU(GB)": _gpu.get(p.pid, 0.0),
                           "켜진 지(분)": int((time.time() - p.info["create_time"]) / 60)} for p in _others])
            .set_index("PID").rename_axis("다른 Jupyter 커널"))
    if CLOSE_OTHER_KERNELS:
        for p in _others:
            p.terminate()
        _gone, _alive = psutil.wait_procs(_others, timeout=8)
        for p in _alive:
            p.kill()
        print(f"🧹 다른 커널 {len(_others)}개 종료 → RAM·GPU 반환", flush=True)
        time.sleep(2)
    else:
        print(f"⚠️ 다른 Jupyter 커널 {len(_others)}개가 RAM {sum(_gb(p.info['memory_info'].rss) for p in _others):.1f}GB · "
              f"GPU {sum(_gpu.get(p.pid, 0) for p in _others):.1f}GB 를 쓰고 있습니다 → 메모리 부족으로 커널이 강제 종료될 수 있음\n"
              f"   끄는 법: 해당 노트북에서 커널 종료(Shut Down) · 터미널 'kill {' '.join(str(p.pid) for p in _others)}' · "
              "또는 위 CLOSE_OTHER_KERNELS = True 후 이 셀 재실행", flush=True)
_vm, _sw = psutil.virtual_memory(), psutil.swap_memory()
print(f"RAM 사용 가능 {_gb(_vm.available)} / {_gb(_vm.total)}GB · 이 커널 {_gb(_me.memory_info().rss)}GB · "
      f"스왑 {_gb(_sw.total)}GB · {gpu_mem()}", flush=True)
if _vm.available < 4 * 1024**3:
    print("⚠️ 사용 가능한 RAM 이 4GB 미만입니다 → 다른 커널을 끄거나, 노트북은 한 번에 하나만 실행하세요", flush=True)
if _sw.total == 0:
    print("ℹ️ 스왑이 없습니다. 메모리가 잠깐 모자랄 때 커널이 바로 죽지 않게 하려면 VM 터미널에서 한 번:\n"
          "   sudo fallocate -l 8G /swapfile && sudo chmod 600 /swapfile && sudo mkswap /swapfile && sudo swapon /swapfile", flush=True)
print(f"⏱ 총 {time.time() - T0:.1f}s")

① torch 로딩…
   완료 5.5s
② 토큰 — OpenAI: .env / HuggingFace: 로그인 정보(hf auth login) 사용, HF_TOKEN 입력창 없음
✔ .env 읽음: /home/spai1322/rfp_rag/.env  (키: OPENAI_API_KEY)


,값,출처,용도
OPENAI_API_KEY,sk-pr…6mQA,.env,gpt-5-mini 생성·RAGAS 심판
HuggingFace,로그인 안 됨,-,모델 다운로드 (KURE-v1·Qwen2.5-VL 은 공개 모델이라 로그인 없이도 ...


③ 토큰 검증 (각 최대 15초)
… OpenAI 키 확인 중
  (2.9s)
검증: {'OPENAI_API_KEY': '✅ 유효 · gpt-5-mini 접근 가능', 'HF_TOKEN': '— 미설정 (공개 모델은 토큰 없이도 받을 수 있음)'}
④ GPU 점검
… GPU 커널 점검 중 (첫 실행은 CUDA 초기화로 수 초~수십 초)


,지금 사용 중인 CPU · GPU · 버전
CPU,Intel(R) Xeon(R) CPU @ 2.20GHz
CPU 코어·스레드,물리 2코어 / 논리 4스레드
RAM,15.6 GB
GPU,NVIDIA L4
GPU 메모리,22.0 GB
GPU 아키텍처,sm_89 · bf16 지원
NVIDIA 드라이버,580.178.04
PyTorch,2.11.0+cu128
CUDA (torch 빌드),12.8
cuDNN,9.19.0


✅ CUDA 커널 실행 OK (fp16/bf16 matmul, softmax) (1.0s)
GPU 메모리: 이 프로세스 0.01GB(예약 0.02GB) · 장치 여유 11.1 / 22.0GB
⑤ 메모리 점검


,RAM(GB),GPU(GB),켜진 지(분)
다른 Jupyter 커널,,,
21121,0.0,0.0,7366
21255,0.0,0.0,7319
26656,0.0,0.0,5885
30472,2.3,7.5,5826
31252,0.0,0.0,5756
39221,2.6,2.9,4473
54510,0.6,0.2,1305
61668,0.1,0.0,6


⚠️ 다른 Jupyter 커널 8개가 RAM 5.6GB · GPU 10.6GB 를 쓰고 있습니다 → 메모리 부족으로 커널이 강제 종료될 수 있음
   끄는 법: 해당 노트북에서 커널 종료(Shut Down) · 터미널 'kill 21121 21255 26656 30472 31252 39221 54510 61668' · 또는 위 CLOSE_OTHER_KERNELS = True 후 이 셀 재실행
RAM 사용 가능 1.8 / 15.6GB · 이 커널 1.2GB · 스왑 0.0GB · GPU 메모리: 이 프로세스 0.01GB(예약 0.02GB) · 장치 여유 11.1 / 22.0GB
⚠️ 사용 가능한 RAM 이 4GB 미만입니다 → 다른 커널을 끄거나, 노트북은 한 번에 하나만 실행하세요
ℹ️ 스왑이 없습니다. 메모리가 잠깐 모자랄 때 커널이 바로 죽지 않게 하려면 VM 터미널에서 한 번:
   sudo fallocate -l 8G /swapfile && sudo chmod 600 /swapfile && sudo mkswap /swapfile && sudo swapon /swapfile
⏱ 총 10.7s


In [2]:
# 0-2) ★★ 챗봇 생성 · 검색 · 실시간 RAGAS 파라미터 (01 의 인덱스만 사용 — 02 실행 불필요) ★★
WORK_DIR = "artifacts"                       # 01 노트북의 WORK_DIR 과 같아야 함

# ─────────────────────────────────────────── ⚖️ 하이브리드 검색 (BM25 : Dense) · top-k
TOP_K        = 5
DENSE_WEIGHT = 0.8                           # Dense 비율 (BM25 = 1 − 0.6 = 0.4)
FUSION       = "score"                       # score(min-max 가중합) | rrf
CANDIDATE_K, RRF_K = 50, 60
HNSW_EF_SEARCH = 128                         # faiss index_type="hnsw" 일 때만 (재구축 불필요)
ROUTING = dict(enabled=True, auto_extract_orgs=True, followup_query="none", orgs={})   # 질문 속 기관명 → 해당 문서로 범위 한정 (사전 없음)

# ─────────────────────────────────────────── 🤖 LLM (답변 생성) — 토큰 절약 핵심 설정
#   LLM_MODEL       : "gpt-5-mini"  (추론 모델, 입력 $0.25 · 출력 $2.00 / 1M 토큰)
#                     "gpt-6-luna"  (가장 싸고 빠른 GPT-6, 입력 $0.10 · 출력 $0.50 / 1M — REASONING_EFFORT="none" 이면 추론 없이 답변)
#   REASONING_EFFORT: none | minimal | low | medium | high | xhigh | max
#                     낮을수록 '생각(추론) 토큰'이 줄어 싸고 빠름. none = 추론 토큰 0 (gpt-6-luna 등 지원 모델만)
#                     모델이 지원하지 않는 값이면 가장 가까운 낮은 값으로 자동 조정 (예: gpt-5-mini 에 none → minimal, xhigh/max → high)
#   ※ 추론 토큰은 '출력 토큰'으로 과금됩니다. 추론 강도 none 일 때만 TEMPERATURE·TOP_P 가 적용됩니다.
LLM_MODEL        = "gpt-5-mini"
REASONING_EFFORT = "low"
TEMPERATURE      = 0.2
TOP_P            = 1.0
MAX_TOKENS       = 2048       # 답변 1개 출력 상한 (추론 토큰 포함) — 낮추면 비용 상한도 내려감
ALLOW_GENERAL    = True       # True: 문서에 없으면 '[문서 외 일반지식]' 표시 후 답변
HISTORY_TURNS    = 3          # 멀티턴: 직전 N턴을 함께 보냄 (줄이면 입력 토큰 절약)

# ─────────────────────────────────────────── 🔎 ① 질문 해석 — GPT 가 질문을 '검색용 질문'으로 다시 씀 (오타·한영 자판·한자·영어·줄임말)
#   원래 질문 + 다시 쓴 질문 N개로 각각 검색 → RRF 로 합침
QUERY_REWRITE            = True
REWRITE_MODEL            = "gpt-5-mini"     # 예) "gpt-6-luna" (더 쌈 · REWRITE_REASONING_EFFORT="none" 가능)
REWRITE_REASONING_EFFORT = "low"            # ★ 질문을 해석할 때 얼마나 '생각'할지: none | minimal | low | medium | high | xhigh | max
                                            #   low = 빠르고 쌈(일반 오타·자판) · high = 꼬인 질문·후속 질문도 꼼꼼히 (추론 토큰↑ = 비용↑)
REWRITE_N                = 2                # 다시 쓴 검색 질문 개수
REWRITE_MAX_TOKENS       = 2048             # 해석 1회 출력 상한 (추론 토큰 포함 — high 면 넉넉히)
REWRITE_HISTORY_TURNS    = 2                # 후속 질문('그럼 그 사업은?') 해석에 참고할 직전 대화 턴 수 (03 챗봇)
ANSWER_WITH_REWRITE      = True             # ★ True: 답변 GPT 에도 '검색용 질문'(1순위)을 원래 질문과 함께 전달 → 오타·자판·줄임말·후속 질문을 바로잡은 뜻으로 답함
                                            #   False: 답변 GPT 에는 원래 질문만 (다시 쓴 질문은 검색에만)
# ─────────────────────────────────────────── 🎯 ③ 리랭커 — 후보를 질문과 1:1 로 다시 채점해 진짜 관련 있는 TOP_K 만 (내 GPU · API 비용 0)
RERANK            = True
RERANK_MODEL      = "BAAI/bge-reranker-v2-m3"   # 다국어(한국어 포함) CrossEncoder · 첫 실행 때 약 2.3GB 다운로드
RERANK_CANDIDATES = 30                          # 리랭커에 넣을 후보 수 (멀티 쿼리 RRF 상위 N개)

# ─────────────────────────────────────────── ⚖️ RAGAS 심판(Judge) — 평가 토큰의 대부분이 여기서 나감
JUDGE_MODEL            = "gpt-5-mini"    # 예) "gpt-6-luna" + JUDGE_REASONING_EFFORT="none" → 평가 비용 크게 절감
JUDGE_REASONING_EFFORT = "low"           # none | minimal | low | medium | high | xhigh | max (자동 조정 동일)
JUDGE_MAX_TOKENS       = 4096

# ─────────────────────────────────────────── 📏 실시간 평가 (화면 표시만 · 파일 저장 없음)
LIVE_RAGAS  = True            # 답변마다 RAGAS(LLM 심판) — False 면 심판 호출 0
FREE_METRICS = True           # 골든 데이터셋(02 의 0-5 가 만든 eval/golden_set.jsonl)에 있는 질문이면 토큰 0 지표 + 정답 자동 사용
GOLDEN_PATH = "eval/golden_set.jsonl"
RAGAS_CFG = dict(metrics=["faithfulness", "answer_relevancy", "context_precision", "answer_correctness"],
                 judge_model=JUDGE_MODEL, judge_reasoning_effort=JUDGE_REASONING_EFFORT, judge_max_tokens=JUDGE_MAX_TOKENS)
PRICES = {"gpt-5-mini": (0.25, 2.00), "gpt-5-nano": (0.05, 0.40), "gpt-6-luna": (0.10, 0.50), "gpt-5.6-luna": (0.20, 1.20), "gpt-4.1-mini": (0.40, 1.60), "gpt-4.1": (2.00, 8.00)}   # (입력, 출력) $/1M 토큰 — 비용 표시용, 가격이 바뀌면 수정
def usd(p_in, p_out, model):
    """토큰 → 달러 (추론 토큰은 출력 토큰에 포함되어 과금). 가격표에 없는 모델이면 None"""
    p = next((v for k, v in sorted(PRICES.items(), key=lambda kv: -len(kv[0])) if str(model).lower().startswith(k.lower())), None)
    return None if p is None else (p_in or 0) / 1e6 * p[0] + (p_out or 0) / 1e6 * p[1]
def fmt_usd(x):
    return "—" if x is None else (f"${x:,.4f}" if x >= 0.01 else f"${x:.6f}")

# ============ 적용: 01 의 build_config.json(파싱·청크·임베딩·FAISS 설정 + 인덱스 경로) + 위 값 ============
from rag.config import load_config, set_by_path
from rag.hw import profile_overrides, profile_name
from rag.pipeline import Pipeline
from rag import notebook_checkpoint as nb
pd.set_option("display.max_colwidth", 120)
CFG = load_config()
CFG["retrieval"].update(top_k=TOP_K, dense_weight=DENSE_WEIGHT, fusion=FUSION, candidate_k=CANDIDATE_K, rrf_k=RRF_K)
CFG["routing"].update(ROUTING)
CFG["llm"].update(model=LLM_MODEL, temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS,
                  reasoning_effort=REASONING_EFFORT, allow_general=ALLOW_GENERAL, history_turns=HISTORY_TURNS)
CFG["rules"]["enabled"] = False              # 규칙·사전 전처리 없음 — 질문 그대로
CFG["evaluation"].update(RAGAS_CFG)
CFG["retrieval"].update(query_rewrite=dict(enabled=QUERY_REWRITE, model=REWRITE_MODEL, effort=REWRITE_REASONING_EFFORT, n=REWRITE_N,
                                            history_turns=REWRITE_HISTORY_TURNS, answer_with_rewrite=ANSWER_WITH_REWRITE),
                        rerank=dict(enabled=RERANK, model=RERANK_MODEL, candidates=RERANK_CANDIDATES))   # 설정이 바뀌면(답변에 검색용 질문 반영 포함) 02 결과 폴더도 새로

# Windows(로컬)에서 만든 artifacts 를 GCP(리눅스)로 옮겨 온 경우: 저장된 경로의 '\' → '/' 로 바꿔야 파일을 찾음
def fix_windows_paths(work_dir):
    import json as _json
    if os.sep == "\\" or not Path(work_dir).exists():
        return 0
    n = 0
    def conv(x):
        nonlocal n
        if isinstance(x, str) and "\\" in x and ("artifacts" in x or x.startswith(str(work_dir))):
            n += 1
            return x.replace("\\", "/")
        if isinstance(x, dict):
            return {k: conv(v) for k, v in x.items()}
        if isinstance(x, list):
            return [conv(v) for v in x]
        return x
    for name in ("active.json", "build_config.json", "manifest.json"):
        p = Path(work_dir) / name
        if p.exists():
            d = _json.loads(p.read_text(encoding="utf-8"))
            d2 = conv(d)
            if d2 != d:
                p.write_text(_json.dumps(d2, ensure_ascii=False, indent=2), encoding="utf-8")
    return n
_n = fix_windows_paths(WORK_DIR)
if _n:
    print(f"ℹ️ Windows 에서 만든 artifacts 감지 → 경로 {_n}개를 리눅스 형식으로 변환했습니다")
pipe = Pipeline.from_build(CFG, WORK_DIR)    # ← 학습 결과 불러오기 (없으면 01 을 실행하라는 오류)
CFG = pipe.cfg
po = profile_overrides(HW)                   # 질문 임베딩 장치만 현재 PC 에 맞춤
CFG["embedding"].update(device=po["embedding.device"], fp16=po["embedding.fp16"])
CFG["faiss"]["hnsw_ef_search"] = HNSW_EF_SEARCH
bi = pipe.build_info
display(pd.DataFrame({"학습 저장 시각": bi["saved_at"], "임베딩 모델(인덱스)": bi["active"].get("embed_model"),
                      "청크 수": (bi["stats"].get("chunk") or {}).get("chunks"),
                      "chunk_size": bi["cfg"]["chunking"]["chunk_size"], "FAISS": bi["cfg"]["faiss"]["index_type"],
                      "HWP 로더": bi["cfg"]["parsing"].get("hwp_loader"), "generate_only": pipe.generate_only}, index=["01 학습 결과"]).T)
print(f"답변 {LLM_MODEL}(추론 {REASONING_EFFORT}) · max_tokens {MAX_TOKENS} · 심판 {JUDGE_MODEL}(추론 {JUDGE_REASONING_EFFORT}) · 실시간 RAGAS {LIVE_RAGAS} · 토큰 0 지표 {FREE_METRICS}")
from rag.llm import effort_table
print(f"🔎 질문 해석 {'켜짐 · ' + REWRITE_MODEL + ' · 추론 ' + REWRITE_REASONING_EFFORT + f' · 검색 질문 {REWRITE_N}개' if QUERY_REWRITE else '꺼짐'}"
      f" · 🎯 리랭커 {RERANK_MODEL + f' (후보 {RERANK_CANDIDATES} → {TOP_K})' if RERANK else '꺼짐'}")
display(effort_table(([("검색 질문 해석", REWRITE_MODEL, REWRITE_REASONING_EFFORT)] if QUERY_REWRITE else [])
                     + [("답변 생성", LLM_MODEL, REASONING_EFFORT), ("RAGAS 심판", JUDGE_MODEL, JUDGE_REASONING_EFFORT)]))

,01 학습 결과
학습 저장 시각,2026-10-04 10:05:40
임베딩 모델(인덱스),nlpai-lab/KURE-v1
청크 수,18041
chunk_size,700
FAISS,flat
HWP 로더,HwpHwpxLoader
generate_only,True


답변 gpt-5-mini(추론 low) · max_tokens 2048 · 심판 gpt-5-mini(추론 low) · 실시간 RAGAS True · 토큰 0 지표 True
🔎 질문 해석 켜짐 · gpt-5-mini · 추론 low · 검색 질문 2개 · 🎯 리랭커 BAAI/bge-reranker-v2-m3 (후보 30 → 5)


,용도,모델,요청 추론 강도,실제 적용,추론 모델,지원 값,비고
0,검색 질문 해석,gpt-5-mini,low,low,예,"minimal, low, medium, high",
1,답변 생성,gpt-5-mini,low,low,예,"minimal, low, medium, high",
2,RAGAS 심판,gpt-5-mini,low,low,예,"minimal, low, medium, high",


## 1. 인덱스 불러오기 (01 결과 — 학습·임베딩 없음)
> 👥 **담당:** 1-1b ①④ 질문 해석 · 답변 (`rag/rewrite.py`) · ②③ 검색 · 리랭킹 (`rag/retrieval.py`)

In [3]:
# 1-1) 인덱스 로드 (FAISS + BM25 + 임베딩 모델 — 질문 임베딩용 추론만) + 생성기
from rag.llm import Generator
retriever = pipe.load_retriever()
generator = Generator(CFG["llm"]["model"], CFG["llm"].get("timeout", 120))
chat = pipe.make_chat(retriever, generator)
print("FAISS 벡터", retriever.store.index.ntotal, "| BM25 청크", len(retriever.by_id), "| 임베딩", pipe.active.get("embed_model"),
      "| 전처리:", "없음(질문 그대로)" if chat.pre is None else "켜짐", "|", gpu_mem())

FAISS 벡터 18041 | BM25 청크 18041 | 임베딩 nlpai-lab/KURE-v1 | 전처리: 없음(질문 그대로) | GPU 메모리: 이 프로세스 0.01GB(예약 0.02GB) · 장치 여유 11.1 / 22.0GB


In [4]:
# 1-1b) 🔎 검색 파이프라인 — ① GPT 질문 해석 → ② (원래 질문 + 다시 쓴 질문) 각각 BM25 + Dense → RRF 병합 → ③ 리랭커
#      → ④ 답변 GPT 에는 [원 질문] + [검색용 질문(GPT 해석 1순위)] 을 함께 보냅니다 (ANSWER_WITH_REWRITE=True)
#      코드는 rag/ 에 있음 — 이 셀은 0-2 설정값으로 조립만 합니다 (02·03 이 같은 코드를 씀)
#        rag/rewrite.py   ① 질문 해석 · ④ 답변에 검색용 질문 전달   (담당: 질문 해석 · 답변 · 챗봇)
#        rag/retrieval.py ② 멀티 쿼리 하이브리드 · ③ 리랭커            (담당: 검색 · 리랭킹)
import importlib
import rag.rewrite, rag.retrieval
importlib.reload(rag.rewrite); importlib.reload(rag.retrieval)     # .py 를 고친 뒤 이 셀만 다시 실행하면 바로 반영
from rag.rewrite import QueryRewriter, install_answer_with_rewrite
from rag.retrieval import SmartRetriever

RW = (QueryRewriter(REWRITE_MODEL, effort=REWRITE_REASONING_EFFORT, n=REWRITE_N, max_tokens=REWRITE_MAX_TOKENS,
                    history_turns=REWRITE_HISTORY_TURNS, price=usd) if QUERY_REWRITE else None)
SR = SmartRetriever(retriever, chat, rewriter=RW, rerank=RERANK, rerank_model=RERANK_MODEL, candidates=RERANK_CANDIDATES)
chat.ret = SR                                   # 이제 chat.ask() 의 검색 = ① → ② → ③
install_answer_with_rewrite(chat, SR, enabled=ANSWER_WITH_REWRITE)   # ④ 답변 GPT 에 검색용 질문 함께 전달

if RERANK:                                      # 디스크·RAM 여유를 확인하고 리랭커를 미리 올림 — 안 되면 리랭커 없이 계속
    _ok, _why = SR.prepare_reranker()
    if _ok:
        print(f"🎯 리랭커 준비: {RERANK_MODEL} ({_why}) · {gpu_mem()}")
    else:
        RERANK = False
        CFG["retrieval"]["rerank"]["enabled"] = False      # 결과 폴더 이름에도 '리랭커 끔'으로 반영
        print(f"⚠️ 리랭커 건너뜀 — {_why} → 리랭커 없이(질문 해석 + 멀티 쿼리 하이브리드) 계속합니다")
print(f"🔎 질문 해석: {'켜짐 · ' + REWRITE_MODEL + ' · 추론 ' + REWRITE_REASONING_EFFORT if QUERY_REWRITE else '꺼짐 (원래 질문만 검색)'}"
      f" · 검색 = 원래 질문 + 다시 쓴 질문 {REWRITE_N if QUERY_REWRITE else 0}개 → RRF → {'리랭커 → ' if RERANK else ''}TOP {TOP_K}")
print(f"🧠 답변 GPT 에 보내는 질문: {'원래 질문 + 검색용 질문(1순위)' if (QUERY_REWRITE and ANSWER_WITH_REWRITE) else '원래 질문만'}"
      + (' (질문 해석이 꺼져 있어 원래 질문만)' if (ANSWER_WITH_REWRITE and not QUERY_REWRITE) else ''))

⚠️ 리랭커 건너뜀 — RAM 여유 1.2GB — 리랭커를 올리려면 4GB 이상 필요 (다른 노트북 커널을 끄세요) → 리랭커 없이(질문 해석 + 멀티 쿼리 하이브리드) 계속합니다
🔎 질문 해석: 켜짐 · gpt-5-mini · 추론 low · 검색 = 원래 질문 + 다시 쓴 질문 2개 → RRF → TOP 5
🧠 답변 GPT 에 보내는 질문: 원래 질문 + 검색용 질문(1순위)


## 2. 챗봇
> 👥 **담당:** 질문 해석 · 답변 · 챗봇 (03 노트북 책임자) — `rag/llm.py`

In [5]:
# 2-1) 챗봇 엔진 + 실시간 RAGAS 함수 (화면 표시 전용 — 파일 저장 없음)
import time, warnings
from IPython.display import Markdown, display, clear_output
from ragas.dataset_schema import SingleTurnSample
from rag.evaluate import METRIC_LABELS, build_metrics, _score, free_metrics, load_golden
from rag import llm as L
warnings.filterwarnings("ignore", message="Unexpected type for token usage")

TURNS = []                 # 이번 세션의 질문·답변·RAGAS (메모리에만)
GOLD = {g["question"]: g for g in load_golden(GOLDEN_PATH)} if FREE_METRICS else {}   # 골든 데이터셋 (없으면 빈 dict)
_METRICS = {}              # RAGAS 지표 객체 캐시 (처음 한 번만 생성)
REF_ONLY = {"answer_correctness", "context_recall", "factual_correctness"}


def _plan(reference):
    """이번 턴에 계산할 (지표 키, 표시 이름) — 정답이 없으면 정답 필요한 지표는 건너뜀"""
    out = []
    for m in RAGAS_CFG["metrics"]:
        if m == "context_precision":
            out.append(("context_precision" if reference else "context_precision_noref", "Context Precision"))
        elif m in REF_ONLY:
            if reference:
                out.append((m, METRIC_LABELS.get(m, m)))
        else:
            out.append((m, METRIC_LABELS.get(m, m)))
    return out


def _metric(key):
    if key not in _METRICS:
        ec = CFG["evaluation"]
        _METRICS.update(build_metrics([key], ec["judge_model"], retriever.embedder, int(ec["judge_max_tokens"]),
                                      ec.get("judge_reasoning_effort")))
    return _METRICS[key]


def new_chat():
    """대화 기록·RAGAS 표 초기화"""
    chat.reset()
    TURNS.clear()


def chat_turn(question, reference=None):
    """질문 1개 → 답변 (대화 기록 유지). 질문은 가공하지 않고 그대로 보냄"""
    r = chat.ask(question)
    g = GOLD.get(question.strip(), {})
    aug = r.get("augmentation") or {}
    info = SR.last or {}
    rw = info.get("rewrite")
    t = dict(n=len(TURNS) + 1, question=question, answer=r["answer"],
             reference=(reference or "").strip() or g.get("reference") or None, key_facts=g.get("key_facts"),
             golden=bool(g), tokens={"입력": aug.get("prompt_tokens"), "출력": aug.get("completion_tokens"),
                                     "추론": aug.get("reasoning_tokens")}, effort=aug.get("reasoning_effort"),
             contexts=[h.get("body", h["text"]) for h in r["hits"]],
             sources=[f"{h['source'][:34]} " + (f"p.{h['page']}" if h.get("page") else (h.get("section") or "")[:30])
                      for h in r["hits"]],
             retrieval_s=r.get("retrieval_s"), llm_s=r.get("llm_s"), total_s=r.get("total_s"),
             warnings=r.get("warnings") or [], route=(r.get("route") or {}).get("mode"), ragas={}, ragas_s=None,
             judge={"calls": 0, "prompt_tokens": 0, "completion_tokens": 0},
             rewrite=rw, queries=info.get("queries") or [question], route2=info.get("route"),
             answer_q=aug.get("답변에 쓴 질문") if aug.get("검색용 질문 반영") else None,
             usd={"해석": (rw.get("usd") or 0) if (rw and rw.get("new")) else 0.0,
                  "답변": usd(aug.get("prompt_tokens") or 0, aug.get("completion_tokens") or 0, LLM_MODEL), "심판": None})
    if FREE_METRICS and (t["reference"] or t["key_facts"]):     # 토큰 0 지표 (로컬 계산)
        t["ragas"].update({k: v for k, v in free_metrics(t["answer"], t["contexts"], t["reference"], t["key_facts"],
                                                         retriever.embedder).items() if v is not None})
    TURNS.append(t)
    return t


def score_turn(t, on_update=None):
    """한 턴의 RAGAS 를 지표 하나씩 계산 — 지표가 끝날 때마다 on_update() 로 화면 갱신"""
    sample = SingleTurnSample(user_input=t["question"], retrieved_contexts=t["contexts"] or [""],
                              response=t["answer"], reference=t["reference"])
    t0 = time.time()
    L.reset_judge_usage()
    for key, label in _plan(t["reference"]):
        if on_update:
            on_update(f"📏 Q{t['n']} RAGAS — {label} 계산 중…")
        t["ragas"][label] = _score(_metric(key), sample)
        t["ragas_s"] = round(time.time() - t0, 1)
        t["judge"] = {k: L.JUDGE_USAGE[k] for k in ("calls", "prompt_tokens", "completion_tokens")}
        if on_update:
            on_update(None)
    t["usd"]["심판"] = usd(t["judge"]["prompt_tokens"], t["judge"]["completion_tokens"], JUDGE_MODEL)
    return t


def turn_cost(t):
    return sum(v or 0 for v in t["usd"].values())


def cost_line(t):
    u = t["usd"]
    return (f"① 질문 해석 {fmt_usd(u['해석'])} · ④ 답변 {fmt_usd(u['답변'])}"
            + (f" · RAGAS 심판 {fmt_usd(u['심판'])}" if u["심판"] is not None else "") + f" → **이번 턴 {fmt_usd(turn_cost(t))}**")


def answer_md(t):
    warn = "".join(f"\n> ⚠️ {w}" for w in t["warnings"])
    src = "<br>".join(f"[{i}] {s}" for i, s in enumerate(t["sources"], 1))
    tk = t["tokens"]
    gold = " · ⭐ 골든 질문(정답 자동)" if t.get("golden") else ""
    rw = t.get("rewrite")
    if rw:
        rq = " · ".join(rw.get("queries") or []) or f"⚠️ {rw.get('error') or '해석 결과 없음'} → 원래 질문만 검색"
        rline = (f"🔎 **질문 해석** ({rw.get('model')} · 추론 {rw.get('effort') or '-'}{'' if rw.get('new') else ' · 저장본'}): {rq}  \n"
                 f"&nbsp;&nbsp;&nbsp;토큰 입력 {rw.get('prompt_tokens', 0):,} · 출력 {rw.get('completion_tokens', 0):,} (추론 {rw.get('reasoning_tokens', 0):,})\n\n")
    else:
        rline = "🔎 질문 해석 꺼짐 — 원래 질문만 검색\n\n"
    rline += (f"🧠 **답변 GPT 가 받은 질문**: [원 질문] {t['question']} ➕ [검색용 질문] **{t['answer_q']}**\n\n" if t.get("answer_q")
              else "🧠 답변 GPT 가 받은 질문: 원래 질문만\n\n")
    return (f"**🤖 답변** · ⏱ 전체 {t['total_s']}s (검색 {t['retrieval_s']}s · 생성 {t['llm_s']}s) · 라우팅 {t.get('route2') or t['route']}{gold}\n\n"
            + rline +
            f"🪙 답변 토큰 입력 {tk['입력']} · 출력 {tk['출력']} (추론 {tk['추론']}) · 추론 강도 {t.get('effort') or '-'}\n\n"
            f"💵 {cost_line(t)}\n\n"
            f"{t['answer']}{warn}\n\n<details><summary>근거 {len(t['sources'])}개</summary>{src}</details>\n\n---")


def ragas_views():
    """(턴별 표, 세션 종합 표) — 실시간 갱신용 Styler"""
    labels = []
    for t in TURNS:
        labels += [k for k in t["ragas"] if k not in labels]
    rows = [{"#": t["n"], "질문": t["question"][:45], **{k: t["ragas"].get(k) for k in labels},
             "응답(s)": t["total_s"], "답변 토큰": (t["tokens"]["입력"] or 0) + (t["tokens"]["출력"] or 0),
             "심판 토큰": t["judge"]["prompt_tokens"] + t["judge"]["completion_tokens"], "RAGAS(s)": t["ragas_s"],
             "해석 $": t["usd"]["해석"], "답변 $": t["usd"]["답변"], "심판 $": t["usd"]["심판"], "합계 $": turn_cost(t)} for t in TURNS]
    usd_cols = ["해석 $", "답변 $", "심판 $", "합계 $"]
    df = pd.DataFrame(rows, columns=["#", "질문"] + labels + ["응답(s)", "답변 토큰", "심판 토큰", "RAGAS(s)"] + usd_cols)
    recs = []
    for k in labels:
        v = pd.to_numeric(df[k], errors="coerce").dropna()
        recs.append({"지표": k, "평균": v.mean() if len(v) else None, "최소": v.min() if len(v) else None,
                     "최대": v.max() if len(v) else None, "계산된 턴": f"{len(v)}/{len(df)}"})
    means = [r["평균"] for r in recs if r["평균"] is not None and r["평균"] == r["평균"]]
    if recs:
        recs.append({"지표": "★ RAGAS 종합 (지표 평균)", "평균": sum(means) / len(means) if means else None,
                     "최소": None, "최대": None, "계산된 턴": ""})
    summ = pd.DataFrame(recs, columns=["지표", "평균", "최소", "최대", "계산된 턴"]).astype(
        {"평균": float, "최소": float, "최대": float})
    tv = (df.style.format("{:.3f}", subset=labels, na_rep="—").format("{:.2f}", subset=["응답(s)"], na_rep="—")
          .format("{:.1f}", subset=["RAGAS(s)"], na_rep="—").format("{:,.0f}", subset=["답변 토큰", "심판 토큰"], na_rep="—")
          .format(lambda x: fmt_usd(x) if x == x and x is not None else "—", subset=usd_cols)
          .background_gradient(subset=labels, cmap="RdYlGn", vmin=0, vmax=1)
          .set_properties(subset=["질문"], **{"text-align": "left"}).hide(axis="index"))
    sv = (summ.style.format("{:.3f}", subset=["평균", "최소", "최대"], na_rep="")
          .background_gradient(subset=["평균"], cmap="RdYlGn", vmin=0, vmax=1).hide(axis="index"))
    return tv, sv


print("✅ 챗봇 준비 완료 · 실시간 RAGAS:", "켜짐" if LIVE_RAGAS else "꺼짐", "· 지표:", RAGAS_CFG["metrics"],
      f"· 골든 질문 {len(GOLD)}개" if FREE_METRICS else "")

✅ 챗봇 준비 완료 · 실시간 RAGAS: 켜짐 · 지표: ['faithfulness', 'answer_relevancy', 'context_precision', 'answer_correctness'] · 골든 질문 0개


In [8]:
# 2-2) 💬 질문하기 — QUESTION 에 질문을 적고 이 셀을 실행(Shift+Enter) → 답변 · RAGAS · 비용이 바로 아래에 나옵니다.
#      질문을 바꿔 다시 실행하면 앞 대화를 기억하고 이어집니다.  새 대화로 시작: 빈 셀에서 new_chat() 실행
#      정답을 알면 REFERENCE 에 적으세요 → Answer Correctness 도 계산 (골든 질문은 자동)
#      RAGAS 채점(LLM 심판)은 0-2 의 LIVE_RAGAS 로 켜고 끔 — False 면 답변만 (심판 비용 $0)
QUESTION  = "너는 누구?"          # 예) "월세액 세액공재 한도가 얼마에요?"
REFERENCE = ""

if not QUESTION.strip():
    print("ℹ️ QUESTION 에 질문을 적고 이 셀을 실행하세요 (비어 있으면 건너뜀 — 모두 실행 때 자동 질문·비용 없음)")
else:
    display(Markdown(f"**🙋 Q{len(TURNS) + 1}. {QUESTION}**"))
    _t = chat_turn(QUESTION, REFERENCE)
    display(Markdown(answer_md(_t)))
    if LIVE_RAGAS:
        _h_s = display(Markdown("📏 RAGAS 준비…"), display_id=True)
        _h_t = display(Markdown(""), display_id=True)
        _h_m = display(Markdown(""), display_id=True)

        def _upd(msg):
            if msg:
                _h_s.update(Markdown(msg))
            else:
                tv, sv = ragas_views()
                _h_t.update(tv)
                _h_m.update(sv)

        score_turn(_t, _upd)
        _h_s.update(Markdown(f"✅ Q{_t['n']} RAGAS 완료 ({_t['ragas_s']}s) — 아래는 이 세션의 턴별 · 종합 RAGAS"))
    display(Markdown(f"💵 {cost_line(_t)} · 세션 누적 {fmt_usd(sum(turn_cost(x) for x in TURNS))}"))

**🙋 Q1. 너는 누구?**

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

**🤖 답변** · ⏱ 전체 37.507s (검색 26.738s · 생성 10.77s) · 라우팅 none

🔎 **질문 해석** (gpt-5-mini · 추론 low): ⚠️ 해석 결과 없음 → 원래 질문만 검색  
&nbsp;&nbsp;&nbsp;토큰 입력 363 · 출력 14 (추론 0)

🧠 답변 GPT 가 받은 질문: 원래 질문만

🪙 답변 토큰 입력 3054 · 출력 813 (추론 704) · 추론 강도 low

💵 ① 질문 해석 $0.000119 · ④ 답변 $0.002390 → **이번 턴 $0.002508**

컨텍스트에 '너는 누구'에 대한 직접적인 정보가 없습니다.

[문서 외 일반지식] 저는 공공기관 제안요청서(RFP) 문서 기반 질의응답을 돕는 질의응답 도우미입니다. 문서에 기재된 내용만 근거로 답변하고, 금액·기간·비율·날짜·자격요건 등은 원문 표기 그대로 인용해 드립니다.
> ⚠️ gpt-5-mini 은(는) 추론 강도가 none 이 아니면 temperature·top_p 를 받지 않아 기본값으로 호출했습니다.

<details><summary>근거 5개</summary>[1] 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp Ⅳ. 기타 > 3. 개인정보 접근 또는 접속 대상자<br>[2] 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구 3. 개인정보 접근 또는 접속 대상자<br>[3] 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구 3. 개인정보 접근 또는 접속 대상자<br>[4] 국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 2. 최근 3년 사업 수행경험 및 실적(10점) : 점<br>[5] 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스 2. 작성기준일자 : 제안일 현재</details>

---

✅ Q1 RAGAS 완료 (33.6s) — 아래는 이 세션의 턴별 · 종합 RAGAS

#,질문,Faithfulness,Answer Relevancy,Context Precision,응답(s),답변 토큰,심판 토큰,RAGAS(s),해석 $,답변 $,심판 $,합계 $
1,너는 누구?,0.250,0.950,0.250,37.51,"3,867","15,626",33.6,$0.000119,$0.002390,—,$0.002508


지표,평균,최소,최대,계산된 턴
Faithfulness,0.250,0.250,0.250,1/1
Answer Relevancy,0.950,0.950,0.950,1/1
Context Precision,0.250,0.250,0.250,1/1
★ RAGAS 종합 (지표 평균),0.483,,,


💵 ① 질문 해석 $0.000119 · ④ 답변 $0.002390 · RAGAS 심판 $0.009024 → **이번 턴 $0.0115** · 세션 누적 $0.0115

In [9]:
# 2-3) 지금까지의 대화 RAGAS + 💵 GPT 비용 + 💬 대화 기록 한눈에 (화면 표시만) · 새 대화로 초기화하려면 new_chat()
if TURNS:
    tv, sv = ragas_views()
    def _sum(f):
        return sum(f(t) or 0 for t in TURNS)
    _rws = [t["rewrite"] for t in TURNS if t.get("rewrite") and t["rewrite"].get("new")]
    cost_df = pd.DataFrame([
        {"단계": "① 질문 해석", "모델 · 추론": f"{REWRITE_MODEL} · {REWRITE_REASONING_EFFORT}" if QUERY_REWRITE else "끔", "호출": len(_rws),
         "입력 토큰": sum(r["prompt_tokens"] for r in _rws), "출력 토큰": sum(r["completion_tokens"] for r in _rws),
         "추론 토큰": sum(r["reasoning_tokens"] for r in _rws), "비용": _sum(lambda t: t["usd"]["해석"])},
        {"단계": "④ 답변 생성", "모델 · 추론": f"{LLM_MODEL} · {REASONING_EFFORT}", "호출": len(TURNS),
         "입력 토큰": _sum(lambda t: t["tokens"]["입력"]), "출력 토큰": _sum(lambda t: t["tokens"]["출력"]),
         "추론 토큰": _sum(lambda t: t["tokens"]["추론"]), "비용": _sum(lambda t: t["usd"]["답변"])},
        {"단계": "RAGAS 심판", "모델 · 추론": f"{JUDGE_MODEL} · {JUDGE_REASONING_EFFORT}", "호출": _sum(lambda t: t["judge"]["calls"]),
         "입력 토큰": _sum(lambda t: t["judge"]["prompt_tokens"]), "출력 토큰": _sum(lambda t: t["judge"]["completion_tokens"]),
         "추론 토큰": None, "비용": _sum(lambda t: t["usd"]["심판"])},
    ])
    cost_df.loc[len(cost_df)] = ["합계", "", *cost_df[["호출", "입력 토큰", "출력 토큰"]].sum().tolist(),
                                 pd.to_numeric(cost_df["추론 토큰"], errors="coerce").sum(), cost_df["비용"].sum()]
    display(Markdown(f"**대화 {len(TURNS)}턴 · 평균 응답 {pd.Series([t['total_s'] for t in TURNS]).mean():.2f}s · "
                     f"턴당 평균 {fmt_usd(cost_df['비용'].iloc[-1] / len(TURNS))}**"),
            cost_df.style.format("{:,.0f}", subset=["호출", "입력 토큰", "출력 토큰", "추론 토큰"], na_rep="—").format(fmt_usd, subset=["비용"]).hide(axis="index"),
            tv, sv)
    print("※ ③ 리랭커·임베딩·BM25 는 내 GPU/CPU 에서 계산 → $0")
    # 대화 전체 기록 — 2-2 에서 한 질문과 답을 모두 다시 보기
    display(Markdown("**💬 대화 기록**\n\n" + "\n\n".join(
        f"<details{' open' if t is TURNS[-1] else ''}><summary><b>🙋 Q{t['n']}. {t['question']}</b></summary>\n\n{answer_md(t)}\n\n</details>"
        for t in TURNS)))
else:
    print("아직 질문이 없습니다 — 2-2 에서 질문하세요")

**대화 1턴 · 평균 응답 37.51s · 턴당 평균 $0.0115**

단계,모델 · 추론,호출,입력 토큰,출력 토큰,추론 토큰,비용
① 질문 해석,gpt-5-mini · low,1,363,14,0,$0.000119
④ 답변 생성,gpt-5-mini · low,1,"3,054",813,704,$0.002390
RAGAS 심판,gpt-5-mini · low,10,"12,702","2,924",—,$0.009024
합계,,12,"16,119","3,751",704,$0.0115


#,질문,Faithfulness,Answer Relevancy,Context Precision,응답(s),답변 토큰,심판 토큰,RAGAS(s),해석 $,답변 $,심판 $,합계 $
1,너는 누구?,0.250,0.950,0.250,37.51,"3,867","15,626",33.6,$0.000119,$0.002390,$0.009024,$0.0115


지표,평균,최소,최대,계산된 턴
Faithfulness,0.250,0.250,0.250,1/1
Answer Relevancy,0.950,0.950,0.950,1/1
Context Precision,0.250,0.250,0.250,1/1
★ RAGAS 종합 (지표 평균),0.483,,,


※ ③ 리랭커·임베딩·BM25 는 내 GPU/CPU 에서 계산 → $0


**💬 대화 기록**

<details open><summary><b>🙋 Q1. 너는 누구?</b></summary>

**🤖 답변** · ⏱ 전체 37.507s (검색 26.738s · 생성 10.77s) · 라우팅 none

🔎 **질문 해석** (gpt-5-mini · 추론 low): ⚠️ 해석 결과 없음 → 원래 질문만 검색  
&nbsp;&nbsp;&nbsp;토큰 입력 363 · 출력 14 (추론 0)

🧠 답변 GPT 가 받은 질문: 원래 질문만

🪙 답변 토큰 입력 3054 · 출력 813 (추론 704) · 추론 강도 low

💵 ① 질문 해석 $0.000119 · ④ 답변 $0.002390 · RAGAS 심판 $0.009024 → **이번 턴 $0.0115**

컨텍스트에 '너는 누구'에 대한 직접적인 정보가 없습니다.

[문서 외 일반지식] 저는 공공기관 제안요청서(RFP) 문서 기반 질의응답을 돕는 질의응답 도우미입니다. 문서에 기재된 내용만 근거로 답변하고, 금액·기간·비율·날짜·자격요건 등은 원문 표기 그대로 인용해 드립니다.
> ⚠️ gpt-5-mini 은(는) 추론 강도가 none 이 아니면 temperature·top_p 를 받지 않아 기본값으로 호출했습니다.

<details><summary>근거 5개</summary>[1] 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp Ⅳ. 기타 > 3. 개인정보 접근 또는 접속 대상자<br>[2] 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구 3. 개인정보 접근 또는 접속 대상자<br>[3] 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구 3. 개인정보 접근 또는 접속 대상자<br>[4] 국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 2. 최근 3년 사업 수행경험 및 실적(10점) : 점<br>[5] 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스 2. 작성기준일자 : 제안일 현재</details>

---

</details>